In [1]:
%%configure -f
{
    "jars": ["https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar", "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar"]
}

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, -1, Finished, Available, Finished, True)

In [2]:
%run manage_files

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 10, Finished, Available, Finished, True)

In [3]:
%run manage_tables

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 16, Finished, Available, Finished, True)

In [4]:
%run transformation

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 26, Finished, Available, Finished, True)

In [5]:
from pyspark.sql.functions import *
from sedona.spark import SedonaContext

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 27, Finished, Available, Finished, False)

In [6]:
NYC_AREA_MIN_LONGITUDE = -75.5
NYC_AREA_MAX_LONGITUDE = -72.0
NYC_AREA_MIN_LATITUDE = 39.0
NYC_AREA_MAX_LATITUDE = 42.5
TAXI_LIMITS = {
    "fare_amount": 3000,
    "total_amount": 5000,
    "tip_amount": 1000,
    "trip_distance": 100
}

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 28, Finished, Available, Finished, False)

In [7]:
TAXI_ZONES_SHAPEFILE_FOLDER_PATH = "Files/NYC_Taxi/taxi_zones/"

YELLOW_TRIPDATA_FOLDER_PATH = "Files/NYC_Taxi/yellow_tripdata"
YELLOW_TRIPDATA_SCHEMA = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "integer",
    "trip_distance": "double",
    "pickup_location_id": "integer",
    "dropoff_location_id": "integer",
    "pickup_longitude": "double",
    "pickup_latitude": "double",
    "dropoff_longitude": "double",
    "dropoff_latitude": "double",
    "fare_amount": "double",
    "total_amount": "double",
    "tip_amount": "double"
}
YELLOW_TRIPDATA_RESULT_COLUMNS = [
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "fare_amount",
    "total_amount",
    "tip_amount"
]
YELLOW_TRIPDATA_COLUMN_MAPPING = {
    "Trip_Pickup_DateTime": "pickup_datetime",
    "tpep_pickup_datetime": "pickup_datetime",
    "Trip_Dropoff_DateTime": "dropoff_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "Passenger_Count": "passenger_count",
    "Trip_Distance": "trip_distance",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Start_Lon": "pickup_longitude",
    "Start_Lat": "pickup_latitude",
    "End_Lon": "dropoff_longitude",
    "End_Lat": "dropoff_latitude",
    "Fare_Amt": "fare_amount",
    "Total_Amt": "total_amount",
    "Tip_Amt": "tip_amount"
}

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 29, Finished, Available, Finished, False)

In [8]:
def get_existing_months(df, date_column):
    return [row["yyyy-mm"] for row in df.select(date_format(date_column, "yyyy-MM").alias("yyyy-mm")).distinct().collect()]

def get_loaded_months(table_name, date_column):
    if not is_table_exist(table_name, "Silver_lakehouse"):
        return []
    df = read_table(table_name, "Silver_lakehouse")
    return get_existing_months(df, date_column)

def get_new_files(file_paths, existing_months):
    return [path for path in file_paths if path[-15:-8] not in existing_months]

def get_new_parquet_files(folder_path, table_name, date_column):
    file_paths = get_parquet_files(folder_path)
    existing_months = get_loaded_months(table_name, date_column)
    return get_new_files(file_paths, existing_months)

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 30, Finished, Available, Finished, False)

In [9]:
def create_sedona_context(spark):
    return SedonaContext.create(spark)

def read_taxi_zones_shapefile(sedona, file_path):
    return sedona.read.format("org.apache.sedona.sql.datasources.shapefile.ShapefileDataSource").load(file_path)

def create_temp_view(df, view_name):
    df.createOrReplaceTempView(view_name)

def transform_taxi_zones(df):
    return df.selectExpr("*", "ST_Transform(geometry, 'EPSG:2263', 'EPSG:4326') AS new_geometry")

def enrich_taxi_locations(df, sedona):
    create_temp_view(df, "trips")
    return sedona.sql("""
        SELECT
            trips.pickup_datetime,
            trips.dropoff_datetime,
            trips.passenger_count,
            trips.trip_distance,
            COALESCE(trips.pickup_location_id, pickup_zone.LocationID) AS pickup_location_id,
            COALESCE(trips.dropoff_location_id, dropoff_zone.LocationID) AS dropoff_location_id,
            trips.fare_amount,
            trips.total_amount,
            trips.tip_amount,
            trips.pickup_longitude,
            trips.pickup_latitude,
            trips.dropoff_longitude,
            trips.dropoff_latitude
        FROM trips
        LEFT JOIN zones AS pickup_zone 
        ON trips.pickup_location_id IS NULL
            AND ST_Contains(pickup_zone.new_geometry, ST_Point(trips.pickup_longitude, trips.pickup_latitude))
        LEFT JOIN zones AS dropoff_zone
        ON trips.dropoff_location_id IS NULL
            AND ST_Contains(dropoff_zone.new_geometry, ST_Point(trips.dropoff_longitude, trips.dropoff_latitude))
    """)

def get_special_location_ids():
    df = read_table("nyc_taxi_zones", "Silver_lakehouse")
    special_zones = {}
    for borough in ["UNKNOWN", "OUTSIDE OF NYC"]:
        zone = df.filter(upper(trim(col("Borough"))) == borough).select("LocationID").first()
        if zone is None:
            raise ValueError(f"Could not find {borough} zone")
        special_zones[borough] = zone["LocationID"]
    return special_zones["UNKNOWN"], special_zones["OUTSIDE OF NYC"]

def load_taxi_zones(sedona, shapefile_folder_path):
    zones_df = read_taxi_zones_shapefile(sedona, shapefile_folder_path)
    zones_df = transform_taxi_zones(zones_df)
    create_temp_view(zones_df, "zones")

def prepare_spatial_processing(shapefile_folder_path):
    sedona = create_sedona_context(spark)
    load_taxi_zones(sedona, shapefile_folder_path)
    return sedona

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 31, Finished, Available, Finished, False)

In [10]:
def is_valid_coordinates(longitude_column, latitude_column):
    return (col(longitude_column).between(NYC_AREA_MIN_LONGITUDE, NYC_AREA_MAX_LONGITUDE) & col(latitude_column).between(NYC_AREA_MIN_LATITUDE, NYC_AREA_MAX_LATITUDE))

def is_valid_taxi_location(location_id, longitude, latitude):
    return (col(location_id).isNotNull() | is_valid_coordinates(longitude, latitude))

def is_zero_coordinates(longitude, latitude):
    return (col(longitude) == 0) & (col(latitude) == 0)

def filter_taxi_coordinates(df):
    is_pickup_valid = is_valid_taxi_location("pickup_location_id", "pickup_longitude", "pickup_latitude")
    is_dropoff_valid = is_valid_taxi_location("dropoff_location_id", "dropoff_longitude", "dropoff_latitude")
    is_pickup_zero = is_zero_coordinates("pickup_longitude", "pickup_latitude")
    is_dropoff_zero = is_zero_coordinates("dropoff_longitude", "dropoff_latitude")
    return df.filter(
        (is_pickup_valid & (is_dropoff_valid | is_dropoff_zero)) |
        (is_dropoff_valid & is_pickup_zero)
    )

def filter_taxi_amounts(df, limits):
    for column, max_value in limits.items():
        df = df.filter(col(column).between(0, max_value))
    return df

def fill_empty_numeric(df, schema, excluded_columns=None):
    excluded_columns = excluded_columns or []
    numeric_columns = [(column, data_type) for column, data_type in schema.items() if data_type in ["integer", "double"] and column not in excluded_columns]
    for column, data_type in numeric_columns:
        if data_type == "double":
            df = df.withColumn(column, nanvl(col(column), lit(0.0)))
    return df.fillna(0, subset=[column for column, _ in numeric_columns])

def normalize_taxi(df, column_mapping, schema):
    df = rename_columns(df, column_mapping)
    df = add_missing_columns(df, schema)
    df = cast_columns(df, schema)
    df = fill_empty_numeric(df, schema, ["pickup_location_id", "dropoff_location_id"])
    df = select_columns(df, schema)
    return df

def filter_taxi(df, limits):
    df = drop_null_rows(df, ["pickup_datetime", "dropoff_datetime"])
    df = filter_taxi_coordinates(df)
    df = filter_taxi_amounts(df, limits)
    return df

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 32, Finished, Available, Finished, False)

In [11]:
def assign_special_location_id(df, location_id, longitude, latitude, unknown_location_id, outside_nyc_location_id):
    return df.withColumn(location_id,
        when(
            col(location_id).isNull() &
            ((col(longitude) == 0.0) | (col(latitude) == 0.0)),
            unknown_location_id
        ).when(
            col(location_id).isNull() &
            (col(longitude) != 0.0) & (col(latitude) != 0.0),
            outside_nyc_location_id
        ).otherwise(col(location_id))
    )

def assign_special_location_ids(df, unknown_location_id, outside_nyc_location_id):
    for coor_type in ["pickup", "dropoff"]:
        df = assign_special_location_id(df, f"{coor_type}_location_id", f"{coor_type}_longitude", f"{coor_type}_latitude", unknown_location_id, outside_nyc_location_id)
    return df

def filter_nyc_trips(df, unknown_location_id, outside_nyc_location_id):
    return df.filter(
        (~col("pickup_location_id").isin(unknown_location_id, outside_nyc_location_id)) |
        (~col("dropoff_location_id").isin(unknown_location_id, outside_nyc_location_id))
    )

def finalize_taxi_locations(df, result_columns):
    unknown_location_id, outside_nyc_location_id = get_special_location_ids()
    df = assign_special_location_ids(df, unknown_location_id, outside_nyc_location_id)
    df = filter_nyc_trips(df, unknown_location_id, outside_nyc_location_id)
    df = select_columns(df, result_columns)
    return df

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 33, Finished, Available, Finished, False)

In [12]:
def process(folder_path):
    table_name = folder_path.rstrip("/").split("/")[-1]
    new_files = get_new_parquet_files(folder_path, table_name, "pickup_datetime")
    if not new_files:
        return
    sedona = prepare_spatial_processing(TAXI_ZONES_SHAPEFILE_FOLDER_PATH)
    for file_path in new_files[:1]:
        df = read_parquet_file(file_path)
        df = normalize_taxi(df, YELLOW_TRIPDATA_COLUMN_MAPPING, YELLOW_TRIPDATA_SCHEMA)
        df = filter_taxi(df, TAXI_LIMITS)
        df = enrich_taxi_locations(df, sedona)
        df = finalize_taxi_locations(df, YELLOW_TRIPDATA_RESULT_COLUMNS)
        # df.explain("formatted")
        save_table(df, table_name, "Silver_lakehouse", "append")


StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 34, Finished, Available, Finished, False)

In [13]:
process(YELLOW_TRIPDATA_FOLDER_PATH)

StatementMeta(, 412de2ba-082f-4d94-83cf-93e8ec3d0100, 35, Submitted, Running, Running, True)